In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("persistingDataApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/17 20:16:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
yellowTaxisDF = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "./Files/YellowTaxis_202210.csv"
)

# Aggregate the data
yellowTaxisGroupedDf = yellowTaxisDF.dropDuplicates().groupBy("PULocationID").agg(sum("total_amount"))

In [6]:
yellowTaxisGroupedDf.write.option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").csv(
    "./Files/Output/CacheTestWithoutEnabling.csv"
)

25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:19:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [7]:
import pyspark

yellowTaxisGroupedDf.persist(pyspark.StorageLevel.MEMORY_AND_DISK)

DataFrame[PULocationID: int, sum(total_amount): double]

In [8]:
yellowTaxisGroupedDf.write.option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").csv(
    "./Files/Output/CacheTestEnabledFirstTime.csv"
)

25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/17 20:23:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [9]:
yellowTaxisGroupedDf.write.option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").csv(
    "./Files/Output/CacheTestEnabledAndCached.csv"
)

In [ ]:
yellowTaxisGroupedDf.unpersist()

DataFrame[PULocationID: int, sum(total_amount): double]

25/06/17 23:07:47 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 160604 ms exceeds timeout 120000 ms
25/06/17 23:07:47 WARN SparkContext: Killing executors is not supported by current scheduler.
25/06/17 23:07:50 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$